# Chapter 13: shocks and exchange rates

Rebuilds the shock table from the synthetic ledger lines (cost split per hL into malt, energy and other) and draws both figures. Lines are rounded to the dollar and totals add the rounded lines, as the book prints them. Metal is taken as 60 percent of the can price.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

lines = pd.read_csv("../data/shock_inputs.csv").set_index("line")
scen = pd.read_csv("../data/shock_scenarios.csv").set_index("scenario")
METAL_SHARE = 0.6
CARRIED = 27.0  # beer in tank is carried at 27 a hL: malt 15.90, energy 4.00, other 7.10


def price_lines(malt_pct, energy_pct, alu_pct):
    m, e, a = 1 + malt_pct / 100, 1 + energy_pct / 100, 1 + METAL_SHARE * alu_pct / 100
    out = {}
    for name, r in lines.iterrows():
        per_hl = r.malt_usd_hl * m + r.energy_usd_hl * e + r.other_usd_hl
        if r.method == "malt_only":
            out[name] = r.base_usd * m
        elif r.method == "follows_carried_cost":
            out[name] = r.base_usd * per_hl / CARRIED
        else:
            out[name] = r.beer_hl * per_hl + r.can_material_usd * a
    return pd.Series(out).round(0)


table = pd.DataFrame({s: price_lines(*scen.loc[s]) for s in scen.index})
table.loc["Total"] = table.sum()
print(table.astype(int))
base, stress, opp = table.loc["Total"]
print(f"Stress adds {stress - base:,.0f} ({100 * (stress - base) / base:.1f} percent); opportunity trims {base - opp:,.0f}")
print(f"Base at book rates: INR {base * 90:,.0f}; EUR {base / 1.1:,.0f}")
print((100 * (table.Stress / table.Base - 1)).round(1).rename("Stress change, percent"))

## Translation against transaction (chapter 13.2)

In [ ]:
eur_before = base / 1.10
eur_after = base * 0.95 / 1.10
print(f"Translation: {eur_before:,.0f} EUR reads {eur_after:,.0f} EUR after the dollar weakens 5 percent: {eur_before - eur_after:,.0f} EUR lower, nothing saved")
DOLLAR_SHARE = 0.40  # synthetic; take yours from the purchase ledger
inr = base * 90
for move in (5,):
    print(f"Transaction: INR {inr:,.0f}; rupee weakens {move} percent: INR {inr * DOLLAR_SHARE * move / 100:,.0f} more ({DOLLAR_SHARE * move:.1f} percent)")

## The probability of banking (chapter 13.4)

In [ ]:
def factor(rate, years=3):
    return sum(1 / (1 + rate) ** t for t in range(1, years + 1))


def value(p, rate, saving=20000, cost=8000):
    return p * saving * factor(rate) - cost


v = value(0.5, 0.12)
print(f"Three years at 12 percent: {20000 * factor(0.12):,.0f}; expected net value {v:,.0f}")
print(f"Rate 16 percent: {value(0.5, 0.16):,.0f} ({100 * (1 - value(0.5, 0.16) / v):.1f} percent lower)")
print(f"Probability 0.7 to 0.5: {value(0.7, 0.12):,.0f} to {v:,.0f} ({100 * (1 - v / value(0.7, 0.12)):.1f} percent lower)")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
bottom = pd.Series(0.0, index=scen.index)
for name in lines.index:
    a1.bar(scen.index, table.loc[name], bottom=bottom, label=name)
    bottom += table.loc[name]
for i, s in enumerate(scen.index):
    a1.text(i, bottom[s] + 800, f"{bottom[s]:,.0f}", ha="center")
a1.set_ylabel("USD a year")
a1.set_title("Figure 13.1: loss total under three scenarios")
a1.legend(fontsize=8)
moves = range(-10, 11)
for share in (0.6, 0.4, 0.2):
    a2.plot(list(moves), [inr * (1 + share * m / 100) / 1e6 for m in moves], label=f"{share:.0%} dollar priced", lw=2 if share == 0.4 else 1)
a2.set_xlabel("Rupee move against the dollar, percent (weaker to the right)")
a2.set_ylabel("Loss, INR million a year")
a2.set_title("Figure 13.2: input driven loss against the exchange rate")
a2.legend()
plt.tight_layout()
plt.show()